In [1]:

import os
from pathlib import Path

repo = Path.home() / "css4007-hw6"

if not repo.exists():
    !git clone https://github.com/turkestanova2006/css4007-hw6.git "{repo}"

%cd {repo}
%pip install -q gymnasium==1.4.0 numpy==1.26.4 pandas



C:\Users\Туркестанова Аида\css4007-hw6
Note: you may need to restart the kernel to use updated packages.


In [5]:

%pip install gymnasium


In [7]:

import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1", map_name="4x4",
               is_slippery=False, render_mode="ansi")

state, info = env.reset(seed=11)

print(env.render())
print("Start state:", state)
print("Gymnasium:", gym.__version__)
print("NumPy:", np.__version__)

env.close()



SFFF
FHFH
FFFH
HFFG

Start state: 0
Gymnasium: 1.4.0
NumPy: 1.26.4


In [5]:

env = gym.make("FrozenLake-v1", map_name="4x4",
               is_slippery=False, render_mode="ansi")

state, info = env.reset(seed=11)
total_reward = 0

for step in range(100):
    action = env.action_space.sample()
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    print(f"Step {step + 1}, State {state}, Reward {reward}")
    print(env.render())

    if terminated or truncated:
        break

print("Total steps:", step + 1)
print("Total reward:", total_reward)

env.close()


Step 1, State 1, Reward 0
  (Right)
SFFF
FHFH
FFFH
HFFG

Step 2, State 5, Reward 0
  (Down)
SFFF
FHFH
FFFH
HFFG

Total steps: 2
Total reward: 0


In [8]:

# Actions: 0=Left, 1=Down, 2=Right, 3=Up

fixed_policy = {
    0: 1, 1: 2, 2: 1, 3: 0,
    4: 1, 6: 1, 8: 2, 9: 1,
    10: 1, 13: 2, 14: 2
}

env = gym.make("FrozenLake-v1", map_name="4x4",
               is_slippery=False, render_mode="ansi")

state, info = env.reset(seed=11)

for step in range(100):
    action = fixed_policy[state]
    state, reward, terminated, truncated, info = env.step(action)

    print(f"Step {step + 1}: State {state}")

    if terminated or truncated:
        break

print("Outcome:", "Goal" if state == 15 else "Hole" if terminated else "Time limit")
print("Steps:", step + 1)

env.close()


Step 1: State 4
Step 2: State 8
Step 3: State 9
Step 4: State 13
Step 5: State 14
Step 6: State 15
Outcome: Goal
Steps: 6


In [9]:

import pandas as pd

results = []
seeds = [11, 22, 33]

for slippery in [False, True]:
    for policy_name in ["Random", "Fixed"]:
        env = gym.make("FrozenLake-v1", map_name="4x4",
                       is_slippery=slippery, max_episode_steps=100)

        for batch in seeds:
            rng = np.random.default_rng(batch)

            for i in range(50):
                episode_seed = 1000 * batch + i
                state, info = env.reset(seed=episode_seed)
                total_reward = 0

                for step in range(100):
                    if policy_name == "Random":
                        action = int(rng.integers(0, 4))
                    else:
                        action = fixed_policy[state]

                    state, reward, terminated, truncated, info = env.step(action)
                    total_reward += reward

                    if terminated or truncated:
                        break

                outcome = ("Goal" if state == 15 else
                           "Hole" if terminated else "Time limit")

                results.append([slippery, policy_name, batch,
                                episode_seed, step + 1,
                                total_reward, outcome])

        env.close()

df = pd.DataFrame(results, columns=[
    "Slippery", "Policy", "Batch seed", "Episode seed",
    "Steps", "Total reward", "Outcome"
])

print("Total episodes:", len(df))
display(df.head())


Total episodes: 600


,Slippery,Policy,Batch seed,Episode seed,Steps,Total reward,Outcome
0,False,Random,11,11000,5,0,Hole
1,False,Random,11,11001,4,0,Hole
2,False,Random,11,11002,13,0,Hole
3,False,Random,11,11003,3,0,Hole
4,False,Random,11,11004,3,0,Hole


In [10]:

summary = df.groupby(["Slippery", "Policy", "Batch seed"]).agg(
    Goals=("Outcome", lambda x: (x == "Goal").sum()),
    Holes=("Outcome", lambda x: (x == "Hole").sum()),
    Time_limits=("Outcome", lambda x: (x == "Time limit").sum()),
    Mean_steps=("Steps", "mean")
).reset_index()

summary["Success %"] = summary["Goals"] / 50 * 100
summary["Mean_steps"] = summary["Mean_steps"].round(2)

display(summary[[
    "Slippery", "Policy", "Batch seed",
    "Goals", "Holes", "Time_limits",
    "Success %", "Mean_steps"
]])


,Slippery,Policy,Batch seed,Goals,Holes,Time_limits,Success %,Mean_steps
0,False,Fixed,11,50,0,0,100.0,6.00
1,False,Fixed,22,50,0,0,100.0,6.00
2,False,Fixed,33,50,0,0,100.0,6.00
3,False,Random,11,0,50,0,0.0,7.94
4,False,Random,22,1,49,0,2.0,7.16
5,False,Random,33,2,48,0,4.0,6.66
6,True,Fixed,11,1,49,0,2.0,4.24
7,True,Fixed,22,2,48,0,4.0,5.48
8,True,Fixed,33,1,49,0,2.0,4.44
9,True,Random,11,1,49,0,2.0,7.98


In [11]:

import os

os.makedirs("outputs", exist_ok=True)

df.to_csv("outputs/part_b_episodes.csv", index=False)
summary.to_csv("outputs/part_b_summary.csv", index=False)

print("Saved 600 episodes and 12 summary rows")


Saved 600 episodes and 12 summary rows


In [12]:

# Save one successful and one failed episode
import json

traces = {}

for wanted in ["Goal", "Hole"]:
    example = df[df["Outcome"] == wanted].iloc[0]
    slippery = bool(example["Slippery"])
    policy_name = example["Policy"]
    seed = int(example["Episode seed"])

    env = gym.make("FrozenLake-v1", map_name="4x4",
                   is_slippery=slippery, render_mode="ansi",
                   max_episode_steps=100)

    state, info = env.reset(seed=seed)
    rng = np.random.default_rng(int(example["Batch seed"]))

    # Reproduce random actions from the start of the batch
    if policy_name == "Random":
        batch_rows = df[
            (df["Slippery"] == slippery) &
            (df["Policy"] == policy_name) &
            (df["Batch seed"] == example["Batch seed"])
        ]
        for _, previous in batch_rows.iterrows():
            if int(previous["Episode seed"]) == seed:
                break
            rng.integers(0, 4, size=int(previous["Steps"]))

    lines = [f"Start (state {state})", env.render()]

    for step in range(100):
        action = (int(rng.integers(0, 4)) if policy_name == "Random"
                  else fixed_policy[state])
        state, reward, terminated, truncated, info = env.step(action)
        lines.append(f"Step {step+1}: action={action}, state={state}, reward={reward}")
        lines.append(env.render())
        if terminated or truncated:
            break

    traces[wanted] = "\n".join(lines)
    env.close()

with open("outputs/part_b_traces.txt", "w", encoding="utf-8") as f:
    for outcome, trace in traces.items():
        f.write(f"=== {outcome} ===\n{trace}\n\n")

print("Saved successful and failed traces")


Saved successful and failed traces


In [13]:

import numpy as np

grid = [
    [".", "M", ".", "G"],
    [".", "#", "I", "H"],
    ["S", ".", ".", "."]
]

rewards = {
    ".": -1, "S": -1, "M": -3,
    "I": -1, "G": 10, "H": -10
}

gamma = 0.9

# Up, Right, Down, Left
actions = [(-1, 0), (0, 1), (1, 0), (0, -1)]

values = np.zeros((3, 4))

print("Original Grid World:")
for row in grid:
    print(" ".join(row))

print("\nInitial values:")
print(values)


Original Grid World:
. M . G
. # I H
S . . .

Initial values:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


In [14]:

# Movement and transition probabilities
def transitions(r, c, action):
    tile = grid[r][c]
    directions = [(action, 1.0)]

    if tile == "I":
        directions = [(action, 0.8),
                      ((action + 1) % 4, 0.1),
                      ((action - 1) % 4, 0.1)]

    result = []
    for direction, prob in directions:
        dr, dc = actions[direction]
        nr, nc = r + dr, c + dc

        if not (0 <= nr < 3 and 0 <= nc < 4) or grid[nr][nc] == "#":
            nr, nc = r, c

        reward = rewards[grid[nr][nc]]
        result.append((prob, nr, nc, reward))

    return result


def value_iteration_step(old_values):
    new_values = old_values.copy()

    for r in range(3):
        for c in range(4):
            if grid[r][c] in ["#", "G", "H"]:
                continue

            q_values = []
            for action in range(4):
                q = sum(
                    prob * (reward + gamma * old_values[nr, nc])
                    for prob, nr, nc, reward in transitions(r, c, action)
                )
                q_values.append(q)

            new_values[r, c] = max(q_values)

    return new_values


v1 = value_iteration_step(values)
v2 = value_iteration_step(v1)

print("V1:")
print(v1)

print("\nRequested V2:")
for r, c in [(0, 1), (0, 2), (1, 2)]:
    print(f"({r+1},{c+1}): {v2[r,c]:.2f}")


V1:
[[-1. -1. 10.  0.]
 [-1.  0. -1.  0.]
 [-1. -1. -1. -1.]]

Requested V2:
(1,2): 8.00
(1,3): 10.00
(2,3): 5.21


In [15]:

# Value Iteration until convergence
v = np.zeros((3, 4))

for iteration in range(1, 10001):
    new_v = value_iteration_step(v)
    difference = np.max(np.abs(new_v - v))
    v = new_v

    if difference < 0.000001:
        break

print("Iterations:", iteration)
print("Final difference:", difference)

if difference >= 0.000001:
    print("Not converged")

print("\nFinal values:")
print(np.round(v, 4))

# Find the best action for each state
arrows = ["↑", "→", "↓", "←"]

print("\nOptimal policy:")
for r in range(3):
    row = []
    for c in range(4):
        if grid[r][c] in ["#", "G", "H"]:
            row.append(grid[r][c])
            continue

        q_values = [
            sum(p * (reward + gamma * v[nr, nc])
                for p, nr, nc, reward in transitions(r, c, a))
            for a in range(4)
        ]

        row.append(arrows[int(np.argmax(q_values))])

    print(" ".join(row))


Iterations: 12
Final difference: 2.1652931048166124e-07

Final values:
[[ 4.2     8.     10.      0.    ]
 [ 2.78    0.      5.8242  0.    ]
 [ 1.5358  2.8176  4.2418  2.8176]]

Optimal policy:
→ → → G
↑ # ↑ H
→ → ↑ ←


In [ ]:

# Extended map with sand and water
grid = [
    [".", "M", ".", "G"],
    [".", "#", "I", "H"],
    ["S", "D", ".", "W"]
]

rewards["D"] = -2
rewards["W"] = -5

v_ext = np.zeros((3, 4))

for iteration_ext in range(1, 10001):
    new_v = value_iteration_step(v_ext)
    diff_ext = np.max(np.abs(new_v - v_ext))
    v_ext = new_v

    if diff_ext < 0.000001:
        break

print("Extended map:")
for row in grid:
    print(" ".join(row))

print("\nIterations:", iteration_ext)
print("Final difference:", diff_ext)
print("Converged:", diff_ext < 0.000001)
print("\nFinal values:")
print(np.round(v_ext, 4))

print("\nOptimal policy:")
for r in range(3):
    row = []
    for c in range(4):
        if grid[r][c] in ["#", "G", "H"]:
            row.append(grid[r][c])
            continue

        q_values = [
            sum(p * (reward + gamma * v_ext[nr, nc])
                for p, nr, nc, reward in transitions(r, c, a))
            for a in range(4)
        ]
        row.append(arrows[int(np.argmax(q_values))])

    print(" ".join(row))


Extended map:
. M . G
. # I H
S D . W

Iterations: 11
Final difference: 2.405881232014906e-07
Converged: True

Final values:
[[ 4.2     8.     10.      0.    ]
 [ 2.78    0.      5.8242  0.    ]
 [ 1.502   2.8176  4.2418  2.8176]]

Optimal policy:
→ → → G
↑ # ↑ H
↑ → ↑ ←


In [15]:

import pandas as pd
import os

original_grid = [
    [".", "M", ".", "G"],
    [".", "#", "I", "H"],
    ["S", ".", ".", "."]
]
extended_grid = [
    [".", "M", ".", "G"],
    [".", "#", "I", "H"],
    ["S", "D", ".", "W"]
]

results_c = []

for map_name, current_grid, V in [
    ("Original", original_grid, v),
    ("Extended", extended_grid, v_ext)
]:
    grid = current_grid

    # Best action for every state
    policy = {}
    for r in range(3):
        for c in range(4):
            if grid[r][c] in ["#", "G", "H"]:
                continue
            q = [
                sum(p * (rew + gamma * V[nr, nc])
                    for p, nr, nc, rew in transitions(r, c, a))
                for a in range(4)
            ]
            policy[r, c] = int(np.argmax(q))

    for batch in [11, 22, 33]:
        for i in range(50):
            seed = 1000 * batch + i
            rng = np.random.default_rng(seed)
            r, c = 2, 0
            total_return = 0
            outcome = "Time limit"

            for step in range(100):
                moves = transitions(r, c, policy[r, c])
                probs = [move[0] for move in moves]
                choice = rng.choice(len(moves), p=probs)
                _, nr, nc, reward = moves[choice]

                total_return += gamma ** step * reward
                r, c = nr, nc

                if grid[r][c] in ["G", "H"]:
                    outcome = "Goal" if grid[r][c] == "G" else "Hole"
                    break

            results_c.append([map_name, batch, seed,
                              step + 1, outcome, total_return])

df_c = pd.DataFrame(results_c, columns=[
    "Map", "Batch", "Seed", "Steps", "Outcome", "Discounted return"
])

summary_c = df_c.groupby(["Map", "Batch"]).agg(
    Goals=("Outcome", lambda x: (x == "Goal").sum()),
    Holes=("Outcome", lambda x: (x == "Hole").sum()),
    Time_limits=("Outcome", lambda x: (x == "Time limit").sum()),
    Mean_steps=("Steps", "mean"),
    Mean_return=("Discounted return", "mean")
).reset_index()

os.makedirs("outputs", exist_ok=True)
df_c.to_csv("outputs/part_c_episodes.csv", index=False)
summary_c.to_csv("outputs/part_c_summary.csv", index=False)

print("Total episodes:", len(df_c))
display(summary_c.round(4))


Total episodes: 300


,Map,Batch,Goals,Holes,Time_limits,Mean_steps,Mean_return
0,Extended,11,50,0,0,5.00,1.5020
1,Extended,22,50,0,0,5.00,1.5020
2,Extended,33,50,0,0,5.00,1.5020
3,Original,11,44,6,0,4.92,1.5211
4,Original,22,45,5,0,5.04,1.6550
5,Original,33,47,3,0,5.10,2.1510


In [4]:

# Check Grid World rules
grid = extended_grid

checks = {
    "Wall": transitions(1, 0, 1)[0][1:3] == (1, 0),
    "Boundary": transitions(0, 0, 3)[0][1:3] == (0, 0),
    "Ice probabilities": abs(sum(x[0] for x in transitions(1, 2, 0)) - 1) < 1e-10,
    "Mud reward": transitions(0, 0, 1)[0][3] == -3,
    "Sand reward": transitions(2, 0, 1)[0][3] == -2,
    "Water reward": transitions(2, 2, 1)[0][3] == -5,
    "Goal reward": transitions(0, 2, 1)[0][3] == 10,
    "Hole reward": transitions(1, 2, 1)[0][3] == -10,
    "Terminal values": v_ext[0, 3] == 0 and v_ext[1, 3] == 0
}

for name, passed in checks.items():
    print(name, "PASS" if passed else "FAIL")

with open("outputs/part_c_checks.txt", "w", encoding="utf-8") as f:
    for name, passed in checks.items():
        f.write(f"{name}: {'PASS' if passed else 'FAIL'}\n")



NameError: name 'extended_grid' is not defined

In [20]:

# Save value maps and policies
def save_map(name, current_grid, V):
    global grid
    grid = current_grid

    lines = [name, "\nValues:"]
    for row in V:
        lines.append(" ".join(f"{x:7.4f}" for x in row))

    lines.append("\nPolicy:")
    for r in range(3):
        row = []
        for c in range(4):
            if grid[r][c] in ["#", "G", "H"]:
                row.append(grid[r][c])
                continue

            q = [
                sum(p * (rew + gamma * V[nr, nc])
                    for p, nr, nc, rew in transitions(r, c, a))
                for a in range(4)
            ]
            row.append(arrows[int(np.argmax(q))])
        lines.append(" ".join(row))

    return "\n".join(lines)

text = save_map("Original map", original_grid, v)
text += "\n\n" + save_map("Extended map", extended_grid, v_ext)

with open("outputs/part_c_maps.txt", "w", encoding="utf-8") as f:
    f.write(text)

print(text)
print("\nSaved: outputs/part_c_maps.txt")


Original map

Values:
 4.2000  8.0000 10.0000  0.0000
 2.7800  0.0000  5.8242  0.0000
 1.5358  2.8176  4.2418  2.8176

Policy:
→ → → G
↑ # ↑ H
→ → ↑ ←

Extended map

Values:
 4.2000  8.0000 10.0000  0.0000
 2.7800  0.0000  5.8242  0.0000
 1.5020  2.8176  4.2418  2.8176

Policy:
→ → → G
↑ # ↑ H
↑ → ↑ ←

Saved: outputs/part_c_maps.txt
